# 모델 업로드/다운로드 — Google Drive 연동

`baseline.ipynb` 학습 산출물(`sketch_base.pt`, HTP 실험군/대조군 `best.pt`, `classes.json`)을
Google Drive 폴더에 올리고, 다른 환경에서 그 폴더로부터 내려받기 위한 노트북이다.

- Drive 폴더: https://drive.google.com/drive/folders/19GjRd7um9xOLwvmrprC9CyX1g1cIv2W2
- **1) 업로드** 셀은 학습을 실행한 PC에서, **2) 다운로드** 셀은 그 외 환경에서 실행한다.
- 둘 다 같은 노트북에 있으니 필요한 섹션만 실행하면 된다.

## 1. 환경 세팅

In [ ]:
# 업로드에는 pydrive2(OAuth 업로드), 다운로드에는 gdown(공개 링크 다운로드)을 쓴다.
%pip install -q gdown pydrive2

In [ ]:
from pathlib import Path
import json

print("환경 세팅 완료.")

## 2. Config  ⚙️

`OUTPUT_ROOT`는 `baseline.ipynb`와 동일한 산출물 위치(`ai/models/htp_yolo/`)를 가리킨다.

In [ ]:
FOLDER_ID  = "19GjRd7um9xOLwvmrprC9CyX1g1cIv2W2"
FOLDER_URL = f"https://drive.google.com/drive/folders/{FOLDER_ID}"

# baseline.ipynb와 동일한 규칙: 절대경로 필수 (상대경로 함정은 HANDOFF.md 참고)
OUTPUT_ROOT   = Path("htp_yolo").resolve()
DOWNLOAD_DIR  = Path("downloaded_models").resolve()

# 다운로드할 날짜 폴더 지정 (YYMMDD 형식, 예: "260722").
#   ""(빈 문자열) 이면 → 오늘 날짜 폴더 우선, 없으면 가장 최근 날짜 폴더 자동 선택.
#   특정 날짜를 넣으면 → 그 날짜 폴더만 받는다(없으면 에러).
# 어느 경우든 DOWNLOAD_DIR에 같은 파일명으로 '덮어쓰기' 되어 항상 최신 상태를 유지한다.
TARGET_DATE = ""

# 로컬 경로 -> Drive에 올릴 파일명
# (실험군 best.pt = HTP 전용 모델 최종 산출물, 대조군은 비교용 참고 자료)
MODEL_FILES = {
    OUTPUT_ROOT / "sketch_base.pt":                                          "sketch_base.pt",
    OUTPUT_ROOT / "runs" / "stageB_from_quickdraw" / "weights" / "best.pt":  "htp_best.pt",
    OUTPUT_ROOT / "runs" / "stageB_from_coco" / "weights" / "best.pt":       "htp_control_best.pt",
    OUTPUT_ROOT / "classes.json":                                            "classes.json",
}

print("Drive 폴더:", FOLDER_URL)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
for local_path, drive_name in MODEL_FILES.items():
    mark = "✓" if local_path.exists() else "✗ (없음)"
    print(f"  {mark}  {local_path.name:24s} -> {drive_name}")

## 3. 업로드 — 로컬 → Drive  📤

**학습을 실행한 PC에서만** 이 섹션을 실행한다.

### 최초 1회 설정: `client_secrets.json`
개인 Google 계정으로 업로드하려면 OAuth 클라이언트가 필요하다 (서비스 계정이 아니라 본인 계정으로 인증).
1. [Google Cloud Console](https://console.cloud.google.com/) → 새 프로젝트 생성
2. "API 및 서비스" → "라이브러리" → **Google Drive API** 사용 설정
3. "OAuth 동의 화면" → User Type: 외부(External) → 테스트 사용자에 본인 이메일 추가
4. "사용자 인증 정보" → "사용자 인증 정보 만들기" → **OAuth 클라이언트 ID** → 애플리케이션 유형: **데스크톱 앱**
5. 다운로드한 JSON을 이 노트북과 같은 폴더에 `client_secrets.json` 이름으로 저장

최초 실행 시 브라우저 인증 창이 뜨고, 이후 실행부터는 `mycreds.txt`에 캐시된 토큰을 재사용한다.

In [ ]:
CLIENT_SECRETS = Path("client_secrets.json")

drive = None
if not CLIENT_SECRETS.exists():
    print("⚠️ client_secrets.json이 없습니다. 위 마크다운 셀의 설정 방법을 먼저 진행하세요.")
    print("   (다운로드만 할 거라면 이 섹션은 건너뛰어도 된다.)")
else:
    from pydrive2.auth import GoogleAuth
    from pydrive2.drive import GoogleDrive

    gauth = GoogleAuth()
    gauth.LoadClientConfigFile(str(CLIENT_SECRETS))
    gauth.LoadCredentialsFile("mycreds.txt")
    if gauth.credentials is None:
        gauth.LocalWebserverAuth()          # 최초 1회: 브라우저 인증
    elif gauth.access_token_expired:
        gauth.Refresh()
    else:
        gauth.Authorize()
    gauth.SaveCredentialsFile("mycreds.txt")  # 다음 실행을 위해 토큰 캐시

    drive = GoogleDrive(gauth)
    print("Drive 인증 완료.")

In [ ]:
def upload_file(local_path: Path, drive_name: str, folder_id: str) -> str:
    # 같은 이름의 파일이 폴더에 이미 있으면 새로 만들지 않고 내용만 갱신한다(재실행 시 중복 방지).
    query = f"'{folder_id}' in parents and title='{drive_name}' and trashed=false"
    existing = drive.ListFile({"q": query}).GetList()
    if existing:
        f = existing[0]
        action = "갱신"
    else:
        f = drive.CreateFile({"title": drive_name, "parents": [{"id": folder_id}]})
        action = "신규 업로드"
    f.SetContentFile(str(local_path))
    f.Upload()
    size_mb = local_path.stat().st_size / 1e6
    print(f"  {action}: {drive_name} ({size_mb:.1f} MB)")
    return f["id"]

if drive is not None:
    for local_path, drive_name in MODEL_FILES.items():
        if local_path.exists():
            upload_file(local_path, drive_name, FOLDER_ID)
        else:
            print(f"  건너뜀(파일 없음): {local_path}")
    print("\n업로드 완료 →", FOLDER_URL)
else:
    print("drive 인증이 안 돼 있어 업로드를 건너뜁니다.")

## 4. 다운로드 — Drive → 로컬  📥

**다른 환경(로컬 PC 등)에서** 이 섹션만 실행하면 된다. 인증이 필요 없다.
단, Drive 폴더가 **"링크가 있는 모든 사용자" 보기 권한**으로 공유돼 있어야 한다.

Drive 폴더(`dodam`)는 날짜별 하위 폴더(`YYMMDD`, 예: `260723`)로 모델을 모아둔다.
- Config의 **`TARGET_DATE = ""`** (기본): **오늘 날짜 폴더**를 우선 받고, 없으면 **가장 최근 날짜 폴더**로 폴백.
- **`TARGET_DATE = "260722"`** 처럼 지정: **그 날짜 폴더만** 받는다(없으면 에러).

받은 파일은 항상 **`downloaded_models/`** 에 같은 이름으로 **덮어쓰기**되어, 이 폴더는 늘 선택한 버전으로 최신화된다.

In [ ]:
import gdown
from datetime import datetime
# gdown 내부 헬퍼: 인증 없이 폴더의 "바로 아래 한 단계" 목록만 가져온다(하위 폴더 이름으로 원하는 날짜 폴더를 찾기 위함).
from gdown.download_folder import (
    _get_session,
    _parse_embedded_folder_view,
    _GoogleDriveFile,
)

TODAY = datetime.now().strftime("%y%m%d")   # 오늘 날짜 폴더명 (예: 260723)


def list_date_folders(folder_id: str):
    """루트 Drive 폴더 바로 아래의 하위 폴더 목록 [(id, name), ...]을 반환한다."""
    sess, _ = _get_session(proxy=None, use_cookies=False, user_agent=None)
    _, children = _parse_embedded_folder_view(sess, folder_id)
    return [
        (cid, cname)
        for cid, cname, ctype in children
        if ctype == _GoogleDriveFile.TYPE_FOLDER
    ]


def resolve_target(folders, target_date: str, today: str):
    """TARGET_DATE가 있으면 그 폴더를, 없으면 오늘→최근 순으로 폴더를 고른다.

    반환: (폴더명, 폴더id, 선택사유)
    """
    by_name = {name: cid for cid, name in folders}

    # 1) 특정 날짜를 지정한 경우: 그 폴더가 반드시 있어야 한다.
    if target_date:
        if target_date not in by_name:
            raise RuntimeError(
                f"지정한 날짜 폴더 '{target_date}'를 찾지 못했습니다. "
                f"발견된 폴더: {sorted(by_name)}"
            )
        return target_date, by_name[target_date], f"지정({target_date})"

    # 2) 미지정 + 오늘 폴더가 있으면 오늘 폴더
    if today in by_name:
        return today, by_name[today], "오늘"

    # 3) 오늘 폴더가 없으면 가장 최근 YYMMDD 폴더로 폴백
    def as_date(name):
        try:
            return datetime.strptime(name, "%y%m%d")
        except ValueError:
            return None   # YYMMDD가 아닌 폴더는 후보에서 제외

    dated = [(as_date(name), name, cid) for cid, name in folders]
    dated = [t for t in dated if t[0] is not None]
    if not dated:
        raise RuntimeError(
            "날짜(YYMMDD) 형식의 하위 폴더를 찾지 못했습니다. "
            f"발견된 폴더: {[n for _, n in folders]}"
        )
    dated.sort(reverse=True)                       # 최신 날짜가 앞으로
    _, name, cid = dated[0]
    return name, cid, "가장 최근(폴백)"


folders = list_date_folders(FOLDER_ID)
print("발견된 날짜 폴더:", [n for _, n in folders] or "(없음)")

target_name, target_id, note = resolve_target(folders, TARGET_DATE, TODAY)
print(f"선택된 폴더: {target_name}  [{note}]  (id={target_id})")

# DOWNLOAD_DIR에 그대로 받아 같은 파일명을 덮어쓴다(항상 선택한 버전으로 최신화).
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
gdown.download_folder(id=target_id, output=str(DOWNLOAD_DIR), quiet=False, use_cookies=False)

print(f"\n선택된 날짜: {target_name}")
print(f"다운로드 위치(덮어쓰기): {DOWNLOAD_DIR}")
print("현재 파일:")
for p in sorted(DOWNLOAD_DIR.glob("*")):
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")

## 5. 정리

| 파일 | 내용 |
|---|---|
| `sketch_base.pt` | 범용 손그림 탐지 모델 (최종 산출물 1) |
| `htp_best.pt` | HTP 전용 탐지 모델 — 실험군(QuickDraw→HTP) (최종 산출물 2) |
| `htp_control_best.pt` | 대조군(COCO→HTP) — 비교 참고용 |
| `classes.json` | HTP 47개 클래스 ↔ id 매핑 (추론 시 필수) |

### 재실행 시 참고
- 업로드 셀은 **같은 이름 파일을 갱신**하므로 여러 번 실행해도 중복 파일이 쌓이지 않는다.
- 다운로드 셀은 `gdown.download_folder`가 폴더 전체를 받아온다. 파일 하나만 필요하면
  `gdown.download(id="<file_id>", output="파일명")`을 개별로 써도 된다(파일 ID는 Drive 웹에서 확인).
- `client_secrets.json`, `mycreds.txt`는 인증 정보이니 **git에 커밋하지 말 것** (`.gitignore`에 `*.json` 관련 규칙이 없다면 추가 확인).